# PLSA Retrieval Validation and Query Study

This experiment evaluates how effectively PLSA retrieves relevant documents compared with TF-IDF and LSI. We divide the dataset into training, validation, and test sets. All three methods use the same vocabulary and candidate documents to ensure a fair comparison.

We use a fixed set of 18 queries across six categories, with three types of queries for each category: direct keywords, paraphrases, and related domain terms. This helps us examine how retrieval performance changes when queries use different wording.

The validation set is used to select the number of latent topics (K) and tune query-likelihood settings, such as background smoothing. After selecting these settings, we evaluate the models on the held-out test set to compare their retrieval performance using metrics such as Precision@10, MAP@10, and MRR.

We also examine query vocabulary coverage to determine how many query terms are present in the training vocabulary and whether missing terms may affect retrieval. Additionally, we perform runtime and memory profiling to measure the computational resources required for PLSA training and inference. This helps us understand not only how well PLSA retrieves documents, but also how efficiently it operates.

All shared implementations are imported from `src/`. Run this notebook from the repository root. Old execution outputs were cleared during refactoring. The final cell writes tables and metadata for `Result.py`; experiment-specific loops remain here.

In [ ]:
# Install from the repository root: pip install -r requirements.txt
# Then: python -m spacy download en_core_web_sm


## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

from src.config import EXPERIMENTS, SUPERGROUPS, CATEGORY_QUERIES, SUPERGROUP_QUERIES, QUERY_BANK
from src.dataset import load_corpus
from src.preprocessing import preprocess_texts, split_train_test, split_train_val_test, vectorize_counts
from src.plsa_model import PLSA, train_plsa_restarts
from src.retrieval import fit_baselines, plsa_query_log_likelihood, background_word_probabilities
from src.metrics import metrics_at_k
from src.experiments import make_mixed_document, average_rank_score
from src.results import export_results
EXPERIMENT = 'Retrieval_Validation_Query_Study'
CONFIG = EXPERIMENTS[EXPERIMENT]


## 2. Load a six-category subset of 20 Newsgroups

We deliberately restrict the corpus to six reasonably distinct categories. This changes the documents used to train PLSA, unlike merely merging the original labels after training. The categories remain the dataset's original labels and no labels are passed to PLSA's EM algorithm.

In [ ]:
# Shared loader; corpus selection remains an explicit experimental choice.
corpus = load_corpus(CONFIG["corpus"], seed=SEED)
documents, labels, target_names = corpus.documents, corpus.labels, corpus.target_names
original_labels = corpus.original_labels
source_category_names = corpus.source_category_names
source_to_supergroup = {category: group for group, members in SUPERGROUPS.items() for category in members}
print("Corpus:", CONFIG["corpus"], "Documents:", len(documents))
for i, label in enumerate(target_names):
    print(label, int((labels == i).sum()))


### Why use a subset?
We selected six categories with more distinct topics from the original dataset to check whether PLSA can identify clearer topics and retrieve relevant documents more effectively. This is a diagnostic experiment, not proof that six is the best number of topics. We will also test different values of K (the number of latent topics learned by PLSA), since the number of dataset categories and the number of learned topics can be different.

In [ ]:
from src.profiling import profile_resources


## 3. Train/test split and preprocessing

Text preprocessing cleans documents by removing unnecessary words and punctuation and converting words to their base forms. The cleaned text is then converted into numerical word-count matrices for PLSA.

In [ ]:
# Split raw text first. Fit vocabulary only on training text.
docs_train, docs_val, docs_test, y_train, y_val, y_test = split_train_val_test(documents, labels, seed=SEED)
docs_train_processed = preprocess_texts(docs_train)
docs_val_processed = preprocess_texts(docs_val)
docs_test_processed = preprocess_texts(docs_test)
count_vectorizer, X_train, X_val, X_test = vectorize_counts(docs_train_processed, docs_val_processed, docs_test_processed)
vocabulary = count_vectorizer.get_feature_names_out()
nonempty_test_mask = np.asarray([bool(doc.strip()) for doc in docs_test_processed])
print("Train/test count matrices:", X_train.shape, X_test.shape)
print("Non-empty test texts:", int(nonempty_test_mask.sum()))



## 4. PLSA Model and EM Algorithm

For document \(d\), word \(w\), and latent topic \(z\):

$$
P(w \mid d) = \sum_{z=1}^{K} P(w \mid z)P(z \mid d)
$$

- $\theta_{d,z} = P(z \mid d)$: document-topic distribution.
- $\phi_{z,w} = P(w \mid z)$: topic-word distribution.
- Each row of $\theta$ and $\phi$ sums to one.

**E-step:** Calculate the probability that topic \(z\) generated word \(w\) in document \(d\).

$$
P(z\mid d,w)=
\frac{P(z\mid d)P(w\mid z)}
{\sum_{z'=1}^{K}P(z'\mid d)P(w\mid z')}
$$

**M-step:** Update the document-topic and topic-word distributions using the expected word counts. Repeat the E-step and M-step until the log-likelihood stops improving.


In [ ]:
# The single EM and fold-in implementation lives in src/plsa_model.py.
from src.plsa_model import PLSA


## 5. Multiple random restarts

EM can produce different results depending on its starting values, so we run the model multiple times and select the one with the highest training log-likelihood. However, we evaluate its retrieval performance separately using test data.

Although our dataset has six categories, we do not have to use K=6. PLSA may combine similar categories or divide one category into multiple topics. We start with K=6 and test other values to find how the number of topics affects performance.

In [ ]:
# train_plsa_restarts is imported from src above.

K = 6
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=5, max_iter=100
)
theta_train = best_plsa.transform()
theta_test = best_plsa.fold_in(X_test, max_iter=100)
print("theta_train:", theta_train.shape)
print("theta_test:", theta_test.shape)


## 6. PLSA Training: Runtime and Memory Profiling

We train PLSA with 20 topics and measure the resources required, including training time and memory usage. We also record the number of iterations completed and the final training log-likelihood to assess the model's training performance.

In [ ]:
def fit_plsa_for_profiling():
    model = PLSA(
        n_topics=20,
        max_iter=100,
        tol=1e-4,
        random_state=SEED + 2026,
        beta=1.0
    )
    return model.fit(X_train)

profiled_plsa, training_measurements = profile_resources(
    fit_plsa_for_profiling
)

training_measurements["operation"] = "PLSA training"
training_measurements["K"] = profiled_plsa.n_topics
training_measurements["iterations_completed"] = profiled_plsa.n_iter_
training_measurements["final_training_log_likelihood"] = (
    profiled_plsa.log_likelihoods[-1]
)

training_resources_df = pd.DataFrame([training_measurements])

display(training_resources_df)

## 7. Inspect topics and convergence

Topic IDs are arbitrary: topic 0 is not inherently sports or politics. Interpret each topic by its highest-probability words, and use category-level averages only as a diagnostic.

In [ ]:
for topic_id, words in best_plsa.top_words(vocabulary, n=12).items():
    print(f"\nTopic {topic_id}:")
    print(", ".join(f"{word} ({prob:.3f})" for word, prob in words))

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(best_plsa.log_likelihoods)+1),
         best_plsa.log_likelihoods, marker="o", markersize=3)
plt.xlabel("EM iteration")
plt.ylabel("Training log-likelihood")
plt.title("PLSA convergence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Controlled topic-boundary experiment

We create artificial documents by mixing words from baseball and gun politics documents. We then use PLSA to estimate the topic proportions and compare them with the original word proportions.

However, the word proportions may not exactly match the estimated topic probabilities because topics can overlap or represent only part of a category. We identify the baseball-related and politics-related topics by checking which topics are most common in documents from each category.

In [ ]:
sports_class = target_names.index("rec.sport.baseball")
politics_class = target_names.index("talk.politics.guns")

class_topic_means = np.vstack([
    theta_train[y_train == c].mean(axis=0)
    for c in range(len(target_names))
])
sports_topic = int(class_topic_means[sports_class].argmax())
politics_topic = int(class_topic_means[politics_class].argmax())

print("Sports-associated topic:", sports_topic)
print("Politics-associated topic:", politics_topic)
if sports_topic == politics_topic:
    print("Warning: both categories map to the same dominant topic; interpret estimates cautiously.")


# make_mixed_document is imported from src above.

boundary_rng = np.random.default_rng(SEED + 77)
ratios = [0.9, 0.7, 0.5, 0.3, 0.1]
boundary_rows = []

for ratio in ratios:
    for replicate in range(20):
        mixed_X = make_mixed_document(
            X_train, y_train, sports_class, politics_class,
            ratio, boundary_rng, n_tokens=300
        )
        estimate = best_plsa.fold_in(mixed_X, max_iter=100)[0]
        boundary_rows.append({
            "source_category_a": "rec.sport.baseball",
            "source_category_b": "talk.politics.guns",
            "actual_sports_word_ratio": ratio,
            "actual_politics_word_ratio": 1-ratio,
            "estimated_sports_topic": estimate[sports_topic],
            "estimated_politics_topic": estimate[politics_topic],
            "replicate": replicate
        })

boundary_df = pd.DataFrame(boundary_rows)
boundary_summary = boundary_df.groupby("actual_sports_word_ratio").agg(
    estimated_sports_mean=("estimated_sports_topic", "mean"),
    estimated_sports_std=("estimated_sports_topic", "std"),
    estimated_politics_mean=("estimated_politics_topic", "mean"),
    estimated_politics_std=("estimated_politics_topic", "std")
).reset_index()
display(boundary_summary)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], "k--", label="Ideal reference")
plt.errorbar(
    boundary_summary["actual_sports_word_ratio"],
    boundary_summary["estimated_sports_mean"],
    yerr=boundary_summary["estimated_sports_std"],
    marker="o", capsize=3, label="Estimated sports-topic probability"
)
plt.xlabel("Actual sports word-token proportion")
plt.ylabel("Estimated sports-associated topic probability")
plt.title("PLSA topic-boundary experiment")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 9. TF-IDF and LSI baselines

All baseline models are trained using only the training documents. LSI converts TF-IDF data into a smaller set of features using SVD. Unlike PLSA, which represents topics using probabilities, LSI represents documents using numerical values in a lower-dimensional space.

In [ ]:
# All baselines use the shared train-only fitting implementation.
tfidf_vectorizer, lsi_model, tfidf_splits, lsi_splits = fit_baselines(
    docs_train_processed, docs_val_processed, docs_test_processed, seed=SEED
)
X_train_tfidf, X_val_tfidf, X_test_tfidf = tfidf_splits
lsi_train, lsi_val, lsi_test = lsi_splits



## 10. Validation-driven retrieval evaluation

Each category has three types of queries: direct keywords, rephrased queries, and related terms. We use category labels to identify relevant documents. The validation set helps us choose the best PLSA settings using AP@10. Once the settings are fixed, we evaluate performance on the test set. All methods use the same queries and documents for a fair comparison.


In [ ]:
query_bank = QUERY_BANK
assert set(query_bank) == set(target_names)
K_GRID, BACKGROUND_GRID, N_RESTARTS, TOP_K = [3, 6, 10, 20], [0.0, 0.01, 0.05], 3, 10
background_probability = background_word_probabilities(X_train)

# metrics_at_k is imported from src above.

def evaluate_split(model, theta_docs, split_name, labels, processed_docs,
                   tfidf_docs, lsi_docs, bg_weight=0.0):
    eligible = np.asarray([bool(x.strip()) for x in processed_docs])
    y_eval, theta = np.asarray(labels)[eligible], theta_docs[eligible]
    tfidf_eval, lsi_eval = tfidf_docs[eligible], lsi_docs[eligible]
    rows = []
    for category, variants in query_bank.items():
        target = target_names.index(category)
        for query_type, query_text in variants:
            query = preprocess_texts([query_text])[0]
            q_count = count_vectorizer.transform([query]).astype(np.float64)
            q_tfidf = tfidf_vectorizer.transform([query])
            q_lsi = lsi_model.transform(q_tfidf)
            q_theta = model.fold_in(q_count, max_iter=100)
            scores = {
                "TF-IDF/VSM": cosine_similarity(q_tfidf, tfidf_eval).ravel(),
                "LSI": cosine_similarity(q_lsi, lsi_eval).ravel(),
                "PLSA cosine": cosine_similarity(q_theta, theta).ravel(),
            }
            scores["PLSA query likelihood"] = plsa_query_log_likelihood(
                q_count, theta, model.phi_, smoothing=1e-12,
                background_probability=background_probability, background_weight=bg_weight
            )
            for method, ranking_scores in scores.items():
                rows.append({"split": split_name, "query_category": category,
                             "query_type": query_type, "query_text": query_text,
                             "method": method, "background_weight": bg_weight,
                             **metrics_at_k(ranking_scores, y_eval, target)})
    return rows


likelihood_choices, cosine_choices, models = [], [], {}
for k_topics in K_GRID:
    model, _ = train_plsa_restarts(
        X_train, n_topics=k_topics, n_restarts=N_RESTARTS,
        max_iter=100, base_seed=SEED + 100 * k_topics
    )
    models[k_topics] = model
    theta_val_candidate = model.fold_in(X_val, max_iter=100)
    for weight in BACKGROUND_GRID:
        rows = evaluate_split(model, theta_val_candidate, "validation", y_val,
                              docs_val_processed, X_val_tfidf, lsi_val, weight)
        ll = [r for r in rows if r["method"] == "PLSA query likelihood"]
        likelihood_choices.append({"K": k_topics, "background_weight": weight,
                                   "MAP@10": np.mean([r["AP@10"] for r in ll]),
                                   "Precision@10": np.mean([r["Precision@10"] for r in ll])})
        if weight == 0:
            cos = [r for r in rows if r["method"] == "PLSA cosine"]
            cosine_choices.append({"K": k_topics,
                                   "MAP@10": np.mean([r["AP@10"] for r in cos]),
                                   "Precision@10": np.mean([r["Precision@10"] for r in cos])})

likelihood_tuning = pd.DataFrame(likelihood_choices)
cosine_tuning = pd.DataFrame(cosine_choices)
display(likelihood_tuning.sort_values(["MAP@10", "Precision@10"], ascending=False))
display(cosine_tuning.sort_values(["MAP@10", "Precision@10"], ascending=False))
best_ll = max(likelihood_choices, key=lambda r: (r["MAP@10"], r["Precision@10"], -r["K"], -r["background_weight"]))
best_cos = max(cosine_choices, key=lambda r: (r["MAP@10"], r["Precision@10"], -r["K"]))
ll_model, cos_model = models[best_ll["K"]], models[best_cos["K"]]
print("Validation-selected PLSA query likelihood:", best_ll)
print("Validation-selected PLSA cosine:", best_cos)


theta_test_ll = ll_model.fold_in(X_test, max_iter=100)
theta_test_cos = cos_model.fold_in(X_test, max_iter=100)
evaluation_rows = []
eligible = np.asarray([bool(x.strip()) for x in docs_test_processed])
y_test_eval = np.asarray(y_test)[eligible]
for category, variants in query_bank.items():
    target = target_names.index(category)
    for query_type, query_text in variants:
        query = preprocess_texts([query_text])[0]
        q_count = count_vectorizer.transform([query]).astype(np.float64)
        q_tfidf = tfidf_vectorizer.transform([query])
        q_lsi = lsi_model.transform(q_tfidf)
        q_cos = cos_model.fold_in(q_count, max_iter=100)
        scores = {
            "TF-IDF/VSM": cosine_similarity(q_tfidf, X_test_tfidf[eligible]).ravel(),
            "LSI": cosine_similarity(q_lsi, lsi_test[eligible]).ravel(),
            "PLSA cosine": cosine_similarity(q_cos, theta_test_cos[eligible]).ravel(),
        }
        scores["PLSA query likelihood"] = plsa_query_log_likelihood(
            q_count, theta_test_ll[eligible], ll_model.phi_, smoothing=1e-12,
            background_probability=background_probability,
            background_weight=best_ll["background_weight"]
        )
        for method, ranking_scores in scores.items():
            evaluation_rows.append({"split": "test", "query_category": category,
                                    "query_type": query_type, "query_text": query_text,
                                    "method": method,
                                    **metrics_at_k(ranking_scores, y_test_eval, target)})

evaluation_df = pd.DataFrame(evaluation_rows)
print("Final held-out test results by method:")
display(evaluation_df.groupby("method")[["Precision@10", "Recall@10", "F1@10", "AP@10", "MRR"]].mean())
print("Test results by query family:")
display(evaluation_df.groupby(["query_type", "method"])[["Precision@10", "AP@10", "MRR"]].mean())


## 11. PLSA Inference: Runtime and Memory Profiling

After training, we use both the cosine-based and query-likelihood PLSA models to estimate topic distributions for test documents. We measure the time taken and memory used by each method, with a maximum of 100 iterations for inference. This helps us compare their computational efficiency and understand the resources needed to process unseen documents.

In [ ]:
_, cosine_inference_measurements = profile_resources(
    cos_model.fold_in, X_test, max_iter=100
)

_, likelihood_inference_measurements = profile_resources(
    ll_model.fold_in, X_test, max_iter=100
)

inference_resources_df = pd.DataFrame([
    {
        "operation": "Cosine model: infer test topic mixtures",
        "K": cos_model.n_topics,
        **cosine_inference_measurements
    },
    {
        "operation": "Query-likelihood model: infer test topic mixtures",
        "K": ll_model.n_topics,
        **likelihood_inference_measurements
    }
])

display(inference_resources_df)

## 12. Query Vocabulary Coverage and Retrieval Analysis

This experiment checks how many words in each query are present in the training vocabulary and how well each retrieval method performs on individual queries. It also compares AP@10 for related-term queries across categories to understand how effectively the models retrieve relevant documents when queries use different words from those in the documents.

In [ ]:

vocabulary = set(count_vectorizer.vocabulary_)
coverage_rows = []
for category, variants in query_bank.items():
    for query_type, query_text in variants:
        processed = preprocess_texts([query_text])[0]
        tokens = processed.split()
        unique_tokens = set(tokens)
        matched = unique_tokens & vocabulary
        coverage_rows.append({
            "query_category": category,
            "query_type": query_type,
            "query_text": query_text,
            "preprocessed_query": processed,
            "unique_query_terms": len(unique_tokens),
            "terms_in_training_vocabulary": len(matched),
            "terms_oov": len(unique_tokens - vocabulary),
            "vocabulary_coverage": len(matched) / len(unique_tokens) if unique_tokens else 0.0,
        })

query_coverage_df = pd.DataFrame(coverage_rows)
print("Query coverage against the training vocabulary:")
display(query_coverage_df.sort_values(["query_category", "query_type"]))

per_query_results = evaluation_df.merge(
    query_coverage_df[
        ["query_category", "query_type", "query_text",
         "unique_query_terms", "terms_in_training_vocabulary",
         "terms_oov", "vocabulary_coverage"]
    ],
    on=["query_category", "query_type", "query_text"],
    how="left",
    validate="many_to_one",
)
print("Held-out test metrics for each query and method:")
display(per_query_results.sort_values(
    ["query_category", "query_type", "method"]
)[[
    "query_category", "query_type", "query_text", "method",
    "vocabulary_coverage", "Precision@10", "AP@10", "MRR"
]])

print("Related-terms AP@10 by category and method:")
display(per_query_results[per_query_results["query_type"] == "related_terms"].pivot(
    index="query_category", columns="method", values="AP@10"
))


## 13. Retrieval Performance by Query Type

We compare PLSA, TF-IDF, and LSI across direct queries, paraphrases, and related-term queries. We also compare AP@10 for each query to identify where PLSA performs better or worse than the baseline methods.

In [ ]:
# Average retrieval performance by query family and method

query_family_summary = (
    evaluation_df
    .groupby(["query_type", "method"])[
        ["Precision@10", "Recall@10", "F1@10", "AP@10", "MRR"]
    ]
    .mean()
    .reset_index()
)

display(
    query_family_summary.sort_values(
        ["query_type", "AP@10"],
        ascending=[True, False]
    )
)

# Compare PLSA cosine with TF-IDF and LSI for each individual query

query_comparison = evaluation_df.pivot_table(
    index=["query_category", "query_type", "query_text"],
    columns="method",
    values="AP@10",
    aggfunc="first"
).reset_index()

query_comparison["PLSA_cosine_minus_TFIDF"] = (
    query_comparison["PLSA cosine"]
    - query_comparison["TF-IDF/VSM"]
)

query_comparison["PLSA_cosine_minus_LSI"] = (
    query_comparison["PLSA cosine"]
    - query_comparison["LSI"]
)

display(
    query_comparison.sort_values(
        "PLSA_cosine_minus_TFIDF",
        ascending=False
    )
)

In [ ]:
test_summary = evaluation_df.groupby("method")[["Precision@10", "Recall@10", "F1@10", "AP@10", "MRR"]].mean()
test_summary.plot(kind="bar", figsize=(10, 5), ylim=(0, 1), rot=0)
plt.title("Held-out test retrieval comparison (18 fixed queries)")
plt.ylabel("Macro-averaged score")
plt.xlabel("Method")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 14. Model-order study: how does K affect the model?

Compare different values of K on the same six-category training set to see how the number of topics affects training convergence and topic separation. We record the training log-likelihood and the number of iterations required for convergence. However, a higher training likelihood does not necessarily mean better retrieval performance on unseen documents.

In [ ]:
k_rows = []
for k_topics in [2, 3, 5, 6, 8, 10, 20]:
    model = PLSA(
        n_topics=k_topics, max_iter=80, tol=1e-4,
        random_state=SEED, beta=1.0
    ).fit(X_train)
    k_rows.append({
        "K": k_topics,
        "iterations": model.n_iter_,
        "train_log_likelihood": model.log_likelihoods[-1],
        "test_documents": X_test.shape[0]
    })

k_df = pd.DataFrame(k_rows)
display(k_df)


In [ ]:
# Tables are exported together with run metadata in the final cell.


## 15. Limitations

1. We use only six categories from the 20 Newsgroups dataset, so the results may be different when tested on all 20 categories.
2. PLSA learns topics without using category labels during training. The labels are used only for splitting the data, selecting documents for experiments, and evaluating retrieval.
3. The topics learned by PLSA may not match the six categories exactly. One category may be split into multiple topics, or similar categories may share a topic.
4. In the mixture experiment, the proportion of baseball and gun-politics words may not exactly match the topic probabilities estimated by PLSA.
5. A model with a higher training log-likelihood does not necessarily perform better when retrieving unseen documents.
6. Our evaluation uses a small, fixed set of queries, and category labels serve as an approximate measure of relevance. Human evaluation of query results could provide a more reliable assessment.
7. Memory and runtime profiling helps us understand the resources required for training and retrieval, but the measurements depend on the machine and environment used. They may differ on other systems or with larger datasets.

## Export this run

Save result tables, convergence, topic words, settings and source hashes. `Result.py` reads these files without repeating model training.

In [ ]:
# This is reached only after all experiment cells complete successfully.
result_directory = export_results(EXPERIMENT, globals())
print("Results saved to:", result_directory)
